In [3]:
import json, os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

ROOT       = Path.cwd()                      # this notebook's folder: week_3/day_12
KB_DIR     = ROOT / 'knowledge_base'         # the KB-*.md / RB-*.md files
KB_JSON    = ROOT / 'kb.json'                # what the notebook writes
CHROMA_DIR = ROOT / 'chroma_day12'           # the vector store

load_dotenv(ROOT.parents[1] / '.env')       # ai-transformation-bootcamp/.env

assert KB_DIR.is_dir(), f"knowledge_base not found at {KB_DIR}"
assert os.getenv('OPENAI_API_KEY'), "OPENAI_API_KEY missing in .env"

client = OpenAI()
print("ROOT   :", ROOT)
print("KB_DIR :", KB_DIR, f"({len(list(KB_DIR.glob('*.md')))} md files)")


ROOT   : d:\AI Transformation Bootcamp Project\ai-transformation-bootcamp\week_3\day_12
KB_DIR : d:\AI Transformation Bootcamp Project\ai-transformation-bootcamp\week_3\day_12\knowledge_base (39 md files)


#the settings, all in one place

In [4]:
MODEL       = 'gpt-4o-mini'
EMBED_MODEL = 'text-embedding-3-small'
TOP_K       = 4
SCORE_FLOOR = 0.35        # measured in Section 4, NOT guessed
NOT_COVERED = 'not covered in the knowledge base'

#the two prices we are paying


In [5]:
print('gpt-4o-mini            $0.15/M input, $0.60/M output')
print('text-embedding-3-small $0.02/M')

gpt-4o-mini            $0.15/M input, $0.60/M output
text-embedding-3-small $0.02/M


# 1. The knowledge base(30 articles and 10 runbooks)

In [7]:
files = sorted(KB_DIR.glob('*.md'))
print(len(files), 'files |', files[0].name, '...', files[-1].name)

40 files | KB-001.md ... RB-10.md


#one file looks like

In [8]:
print((KB_DIR / 'KB-001.md').read_text(encoding='utf-8')[:260])

---
id: KB-001
title: "VPN disconnects every few minutes"
type: article
status: current
last_reviewed: 2026-08-20
---

Users on the Cisco AnyConnect client may see the tunnel drop at regular intervals. The most common cause is an MTU mismatch after a wireless 


#the six documents that are deliberately wrong

In [9]:
import yaml

flagged = []
for f in files:
    fm = yaml.safe_load(f.read_text(encoding='utf-8').split('---')[1])
    if fm.get('status') in ('outdated', 'contradictory'):
        flagged.append(fm)
        print(f"{fm['id']:7} {fm['status']:14} conflicts_with={fm['conflicts_with']:7} "
              f"reviewed={fm['last_reviewed']}")
print(f'\n{len(flagged)} of {len(files)} flagged')

KB-003  outdated       conflicts_with=RB-10   reviewed=2023-06-02
KB-006  contradictory  conflicts_with=KB-025  reviewed=2026-08-11
KB-012  contradictory  conflicts_with=RB-06   reviewed=2026-07-30
KB-025  outdated       conflicts_with=KB-006  reviewed=2023-02-09
RB-06   outdated       conflicts_with=KB-012  reviewed=2022-11-18
RB-10   contradictory  conflicts_with=KB-003  reviewed=2026-08-04

6 of 40 flagged


#parse one file into front matter and body

In [14]:
def parse(path):
    _, fm, body = path.read_text(encoding='utf-8').split('---', 2)
    meta = yaml.safe_load(fm)
    for field in ('id', 'title', 'type', 'status'):
        if not meta.get(field):
            raise ValueError(f"{path.name}: front matter '{field}' is missing or empty")
    meta['body'] = body.strip()
    return meta

d = parse(files[0]); print(d['id'], '|', d['status'], '|', len(d['body'].split()), 'words')

KB-001 | current | 104 words


The four validation lines above live **inside** `parse()` — `if not meta.get(field)` catches a
front-matter key that exists but has an empty value (YAML parses `status:` with nothing after it
as `None`), which `field not in meta` would miss. This is not a standalone cell.

#Markdown in, records out

In [15]:
def build():
    """All 40 files -> [{id, type, title, text, status, last_reviewed, conflicts_with}]."""
    docs = [parse(p) for p in sorted(KB_DIR.glob('*.md'))]
    ordered = ([d for d in docs if d['type'] == 'article'] +
               [d for d in docs if d['type'] == 'runbook'])
    return [{'id': d['id'], 'type': d['type'], 'title': d['title'],
             'text': f"{d['title']}. {d['body']}",
             'status': d['status'], 'last_reviewed': str(d['last_reviewed']),
             'conflicts_with': d.get('conflicts_with', '')} for d in ordered]

docs = build()
json.dump(docs, open(KB_JSON, 'w', encoding='utf-8'), indent=1, ensure_ascii=False)
print(len(docs), 'docs |', sum(1 for d in docs if d['type'] == 'article'), 'articles')


40 docs | 30 articles


#build it and check the size

In [16]:
words = [len(d['text'].split()) for d in docs]
print(f'{len(docs)} docs | words: min={min(words)} mean={sum(words)/len(words):.1f} max={max(words)}')

40 docs | words: min=87 mean=103.0 max=128


#the same thing in tokens, which is what we actually pay for

In [17]:
import tiktoken
enc = tiktoken.get_encoding('cl100k_base')
t = [len(enc.encode(d['text'])) for d in docs]
print(f'tokens/doc: min={min(t)} mean={sum(t)/len(t):.1f} max={max(t)} total={sum(t)}')
print(f'one-off ingest cost @ $0.02/M = ${sum(t)/1e6*0.02:.6f}')

tokens/doc: min=103 mean=124.5 max=160 total=4982
one-off ingest cost @ $0.02/M = $0.000100


#why about 125 tokens per document is a choice, not an accident

In [18]:
print(f'context sent per query = TOP_K x mean_doc = 4 x {sum(t)/len(t):.0f} = {4*sum(t)/len(t):.0f} tokens')
print(f'if docs were 500 tokens each: 4 x 500 = 2000 tokens per query, 4x the bill')

context sent per query = TOP_K x mean_doc = 4 x 125 = 498 tokens
if docs were 500 tokens each: 4 x 500 = 2000 tokens per query, 4x the bill


# 2. Load it into Chroma

#turn text into vectors, in one batch

In [19]:
def _embed(texts):
    r = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in r.data]

#create the collection

In [22]:
import chromadb
chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))
col = chroma.get_or_create_collection('deepa', metadata={'hnsw:space': 'cosine'})

In [23]:
col = chroma.get_or_create_collection('kb_day12', metadata={'hnsw:space': 'cosine'})
col

Collection(name=kb_day12)

#add the documents, with their labels


In [24]:
col.add(ids=[d['id'] for d in docs],
        embeddings=_embed([d['text'] for d in docs]),
        documents=[d['text'] for d in docs],
        metadatas=[{'title': d['title'], 'type': d['type']} for d in docs])
col.count()

40

#make loading safe to run twice

In [25]:
def reindex():
    """Drop and rebuild the collection from the current `docs`. Costs one embedding call."""
    global col
    try:
        chroma.delete_collection('kb_day12')
    except Exception:
        pass
    col = chroma.get_or_create_collection('kb_day12', metadata={'hnsw:space': 'cosine'})
    col.add(ids=[d['id'] for d in docs],
            embeddings=_embed([d['text'] for d in docs]),
            documents=[d['text'] for d in docs],
            metadatas=[{'title': d['title'], 'type': d['type'],
                        'status': d['status'], 'conflicts_with': d['conflicts_with']} for d in docs])
    return col.count()


# 3. Search, and what the score really means

#run a query and look at the raw shape

In [26]:
r = col.query(query_embeddings=_embed(['VPN drops every few minutes']), n_results=4)
r['ids'][0], [round(d, 4) for d in r['distances'][0]]

(['KB-001', 'KB-007', 'KB-011', 'KB-013'], [0.3556, 0.5463, 0.5977, 0.5989])

#convert it into something that points the obvious way

In [27]:
def retrieve(question, k=TOP_K):
    r = col.query(query_embeddings=_embed([question]), n_results=k)
    return [(i, m['title'], d, 1.0 - dist) for i, m, d, dist
            in zip(r['ids'][0], r['metadatas'][0], r['documents'][0], r['distances'][0])]

#check it finds the right thing

In [28]:
for i, title, _, rel in retrieve('What does error APP-771 mean?'):
    print(f'  {rel:.4f}  {i}  {title[:50]}')

  0.5013  KB-010  Application crashes on login with error APP-771
  0.3377  KB-017  Mobile device shows 'device not compliant'
  0.3292  RB-10  Runbook: Device replacement for hardware failure
  0.3254  KB-018  Browser reports certificate error on internal site


 #a question the knowledge base cannot answer


In [29]:
for i, title, _, rel in retrieve('Which AnyConnect client version patches Heartbleed?'):
    print(f'  {rel:.4f}  {i}  {title[:50]}')

  0.4909  KB-001  VPN disconnects every few minutes
  0.2814  KB-011  Wi-Fi drops when moving between floors
  0.2181  KB-006  Multi-factor authentication push notification neve
  0.2156  KB-018  Browser reports certificate error on internal site


# 4. Picking the cut-off score

#the test questions, defined before we score them

In [32]:
ANSWERABLE = [
  ('A1',  'My VPN drops every few minutes. What MTU should I set on the client?',        'KB-001'),
  ('A2',  'Outlook keeps prompting me for a password. What should I clear first?',       'KB-002'),
  ('A3',  'What does error APP-771 mean and how is it fixed?',                           'KB-010'),
  ('A4',  'A user wants a quarantined file released that arrived by email. Can the Service Desk do that?', 'KB-021'),
  ('A5',  "A laptop's disk encryption recovery key is missing from the console. What are the options?",    'KB-027'),
  ('A6',  "What happens to a leaver's mailbox during offboarding?",                      'RB-03'),
  ('A7',  'Who has the authority to declare a major incident?',                          'RB-05'),
  ('A8',  'A user entered their credentials into a phishing page. What is the first action?', 'RB-04'),
  ('A9',  'Why do our database queries time out at month-end?',                          'KB-015'),
  ('A10', 'A guest says their Wi-Fi voucher from yesterday does not work. Why?',         'KB-026'),
]
UNANSWERABLE = [
  ('U1', 'What is the contractual SLA resolution time in hours for a P1 incident?',      None),
  ('U2', 'How do I request a refund for a software licence purchased on expenses?',      None),
  ('U3', 'What is the maximum mailbox size quota in gigabytes for a standard user?',     None),
  ('U4', 'Which AnyConnect client version patches the Heartbleed vulnerability?',        None),
  ('U5', 'How many days of annual leave does a new starter receive?',                    None),
]

QUESTIONS = ANSWERABLE + UNANSWERABLE
print(len(ANSWERABLE), 'answerable +', len(UNANSWERABLE), 'unanswerable =', len(QUESTIONS))

10 answerable + 5 unanswerable = 15


#score every test question, both kinds

In [33]:
rows = [('answerable', qid, retrieve(q)[0][3], retrieve(q)[0][0]) for qid, q, _ in ANSWERABLE]
rows += [('UNANSWERABLE', qid, retrieve(q)[0][3], retrieve(q)[0][0]) for qid, q, _ in UNANSWERABLE]

#the finding: the two groups overlap

In [34]:
a = [r[2] for r in rows if r[0] == 'answerable']; u = [r[2] for r in rows if r[0] == 'UNANSWERABLE']
print(f'answerable  : min={min(a):.4f} mean={sum(a)/len(a):.4f} max={max(a):.4f}')
print(f'unanswerable: min={min(u):.4f} mean={sum(u)/len(u):.4f} max={max(u):.4f}')
print('CLEAN GAP' if min(a) > max(u) else 'OVERLAP — no threshold separates them')

answerable  : min=0.4980 mean=0.6460 max=0.7358
unanswerable: min=0.3110 mean=0.4094 max=0.4985
OVERLAP — no threshold separates them


#so what is the floor worth? Sweep it

In [35]:
for th in (0.25, 0.30, 0.35, 0.40, 0.45, 0.50):
    print(f'floor={th:.2f}: answerable passing={sum(x>=th for x in a)}/10  unanswerable passing={sum(x>=th for x in u)}/5')

floor=0.25: answerable passing=10/10  unanswerable passing=5/5
floor=0.30: answerable passing=10/10  unanswerable passing=5/5
floor=0.35: answerable passing=10/10  unanswerable passing=4/5
floor=0.40: answerable passing=10/10  unanswerable passing=3/5
floor=0.45: answerable passing=10/10  unanswerable passing=1/5
floor=0.50: answerable passing=9/10  unanswerable passing=0/5


# 5. The cited-answer contract

#the schema does three jobs at once

In [36]:
from pydantic import BaseModel, Field
from typing import List

class CitedAnswer(BaseModel):
    answerable: bool = Field(description="True ONLY if the context fully answers the question.")
    answer: str = Field(description="The answer, or the exact phrase 'not covered in the knowledge base'.")
    citations: List[str] = Field(description="IDs of context documents actually used, e.g. ['KB-001'].")

#the prompt

In [37]:
SYSTEM = f"""You answer IT service-desk questions using ONLY the numbered context documents.
Cite the document ID for each claim, inline, like [KB-001].
If the context does not answer the question, set answerable=false and set answer
to exactly: "{NOT_COVERED}\""""

print(SYSTEM)

You answer IT service-desk questions using ONLY the numbered context documents.
Cite the document ID for each claim, inline, like [KB-001].
If the context does not answer the question, set answerable=false and set answer
to exactly: "not covered in the knowledge base"


#one question and its search results, for the next two cells

In [38]:
question = 'My VPN drops every few minutes. What MTU should I set?'
hits = retrieve(question)
print([h[0] for h in hits])

['KB-001', 'KB-007', 'KB-011', 'KB-013']


#number the context so citations have something to point at


In [40]:
ctx = "\n\n".join(f'[{i}] {t}\n{txt}' for i, t, txt, _ in hits)

#one call, structured

In [41]:
r = client.responses.parse(model=MODEL, text_format=CitedAnswer,
    input=[{'role': 'system', 'content': SYSTEM},
           {'role': 'user', 'content': f'Context documents:\n\n{ctx}\n\nQuestion: {question}'}])
r.output_parsed

CitedAnswer(answerable=True, answer='Set the client MTU to 1300 on the affected profile.', citations=['KB-001'])

# 6. The LangGraph pipeline

#the state

In [42]:
from typing import TypedDict, Optional

class State(TypedDict):
    question: str; hits: list; best_score: float
    answer: Optional[str]; citations: List[str]
    abstained: bool; gate: str; in_tok: int; out_tok: int

#node 1: search

In [43]:
def node_retrieve(state):
    state['hits'] = retrieve(state['question'])
    state['best_score'] = state['hits'][0][3] if state['hits'] else 0.0
    return state

#the router: check 1, and it is free

In [44]:
def route_after_retrieve(state):
    return 'generate' if state['best_score'] >= SCORE_FLOOR else 'abstain'

#node 2: generate, with check 2 inside it

In [45]:
def node_generate(state):
    """One structured LLM call, then check 2: the model's own answerable flag."""
    ctx = '\n\n'.join(f'[{i}] {t}\n{txt}' for i, t, txt, _ in state['hits'])
    r = client.responses.parse(
        model=MODEL, text_format=CitedAnswer,
        input=[{'role': 'system', 'content': SYSTEM},
               {'role': 'user', 'content': f"Context documents:\n\n{ctx}\n\nQuestion: {state['question']}"}])
    p = r.output_parsed
    state['in_tok'], state['out_tok'] = r.usage.input_tokens, r.usage.output_tokens
    if p is None or not p.answerable:                    # check 2: the model's own judgement
        state['answer'], state['citations'] = NOT_COVERED, []
        state['abstained'], state['gate'] = True, 'model'
    else:
        state['answer'], state['citations'] = p.answer, p.citations
        state['abstained'], state['gate'] = False, 'answered'
    return state

#node 3: refuse

In [46]:
def node_abstain(state):
    state['answer'], state['citations'] = NOT_COVERED, []
    state['abstained'], state['gate'] = True, 'retrieval'
    state['in_tok'] = state['out_tok'] = 0
    return state

##assemble the graph — nodes, entry point, edges

In [47]:
from langgraph.graph import StateGraph, END
g = StateGraph(State)
g.add_node('retrieve', node_retrieve); g.add_node('generate', node_generate); g.add_node('abstain', node_abstain)
g.set_entry_point('retrieve')
g.add_conditional_edges('retrieve', route_after_retrieve, {'generate': 'generate', 'abstain': 'abstain'})
g.add_edge('generate', END); g.add_edge('abstain', END); graph = g.compile()

In [48]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	generate(generate)
	abstain(abstain)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	retrieve -.-> abstain;
	retrieve -.-> generate;
	abstain --> __end__;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



#answer() runs the graph; cost() prices one call

In [49]:
def answer(question):
    """Run the whole pipeline. Returns the final State dict.

    Always has the same keys, whether it answered or refused. Branch on
    state['abstained'], never on whether a field is present.
    """
    return graph.invoke({'question': question, 'hits': [], 'best_score': 0.0,
                         'answer': None, 'citations': [], 'abstained': False,
                         'gate': '', 'in_tok': 0, 'out_tok': 0})

def cost(in_tok, out_tok):
    """US dollars for one gpt-4o-mini call."""
    return in_tok / 1e6 * 0.15 + out_tok / 1e6 * 0.60

# 7. Fifteen questions

#the ten answerable, and their expected sources

In [50]:
ANSWERABLE = [
  ('A1',  'My VPN drops every few minutes. What MTU should I set on the client?',        'KB-001'),
  ('A2',  'Outlook keeps prompting me for a password. What should I clear first?',       'KB-002'),
  ('A3',  'What does error APP-771 mean and how is it fixed?',                           'KB-010'),
  ('A4',  'A user wants a quarantined file released that arrived by email. Can the Service Desk do that?', 'KB-021'),
  ('A5',  "A laptop's disk encryption recovery key is missing from the console. What are the options?",   'KB-027'),
  ('A6',  "What happens to a leaver's mailbox during offboarding?",                      'RB-03'),
  ('A7',  'Who has the authority to declare a major incident?',                          'RB-05'),
  ('A8',  'A user entered their credentials into a phishing page. What is the first action?', 'RB-04'),
  ('A9',  'Why do our database queries time out at month-end?',                          'KB-015'),
  ('A10', 'A guest says their Wi-Fi voucher from yesterday does not work. Why?',         'KB-026'),
]

#the five unanswerable, and why each one is hard

In [51]:
UNANSWERABLE = [
  ('U1', 'What is the contractual SLA resolution time in hours for a P1 incident?',      None),
  ('U2', 'How do I request a refund for a software licence purchased on expenses?',      None),
  ('U3', 'What is the maximum mailbox size quota in gigabytes for a standard user?',     None),
  ('U4', 'Which AnyConnect client version patches the Heartbleed vulnerability?',        None),
  ('U5', 'How many days of annual leave does a new starter receive?',                    None),
]

#run all fifteen

In [52]:
import pandas as pd

def run_eval():
    """Run all 15 questions. One row per question.

    cite_ok is None for unanswerable questions -- there is no correct citation
    to check, and scoring them False would be wrong.
    """
    out = []
    for qid, q, expected in QUESTIONS:
        s = answer(q)
        out.append({'id': qid, 'kind': 'answerable' if expected else 'UNANSWERABLE',
                    'best_score': round(s['best_score'], 4), 'gate': s['gate'],
                    'abstained': s['abstained'], 'expected': expected or '-',
                    'cited': ','.join(s['citations']) if s['citations'] else '-',
                    'cite_ok': (expected in s['citations']) if expected else None,
                    'cost': cost(s['in_tok'], s['out_tok'])})
    return pd.DataFrame(out)

pd.set_option('display.width', 200)
df = run_eval()
print(df.drop(columns=['cost']).to_string(index=False))

 id         kind  best_score      gate  abstained expected  cited cite_ok
 A1   answerable      0.6660  answered      False   KB-001 KB-001    True
 A2   answerable      0.7340  answered      False   KB-002 KB-002    True
 A3   answerable      0.4980  answered      False   KB-010 KB-010    True
 A4   answerable      0.6101  answered      False   KB-021 KB-021    True
 A5   answerable      0.6958  answered      False   KB-027 KB-027    True
 A6   answerable      0.6586  answered      False    RB-03  RB-03    True
 A7   answerable      0.6058  answered      False    RB-05  RB-05    True
 A8   answerable      0.5387  answered      False    RB-04  RB-04    True
 A9   answerable      0.7358  answered      False   KB-015 KB-015    True
A10   answerable      0.7175  answered      False   KB-026 KB-026    True
 U1 UNANSWERABLE      0.4469     model       True        -      -    None
 U2 UNANSWERABLE      0.3651     model       True        -      -    None
 U3 UNANSWERABLE      0.3110 retrieval

#the result table

In [53]:
ans, una = df[df.kind=='answerable'], df[df.kind=='UNANSWERABLE']
print(f'answered {(~ans.abstained).sum()}/10, cited the expected source {ans.cite_ok.sum()}/10')
print(f'unanswerable answered anyway: {(~una.abstained).sum()}/5')

answered 10/10, cited the expected source 10/10
unanswerable answered anyway: 0/5


# 8. The probe

#the number

In [54]:
print(f'*** {(~una.abstained).sum()} of 5 ***')

*** 0 of 5 ***


#one run is not a result, so repeat it

In [55]:
for run in range(3):
    print(f'run {run}: leaked {sum(1 for _, q, _ in UNANSWERABLE if not answer(q)["abstained"])}/5')

run 0: leaked 0/5
run 1: leaked 0/5
run 2: leaked 0/5


#count the refusals by gate: retrieval or model

In [56]:
print(una.gate.value_counts())

gate
model        4
retrieval    1
Name: count, dtype: int64


#what the 0/5 costs: refusing too often, and a stricter prompt makes it worse

In [57]:
print(f'false abstentions (answerable, refused): {ans.abstained.sum()}/10  -> A4')

false abstentions (answerable, refused): 0/10  -> A4


In [58]:
SYSTEM_STRICT = f"""You answer IT service-desk questions using ONLY the numbered context documents.
Cite the document ID for each claim, inline, like [KB-001].
If the context does not answer the question, set answerable=false and set answer
to exactly: "{NOT_COVERED}"

Context on the same topic is NOT enough. The context must state the answer explicitly.
Never guess a number, a version or a threshold that is not written in the context."""

#ask a question the knowledge base answers twice, differently


In [59]:
mfa_q = 'A user has a hardware MFA token that is rejecting codes. Can I resynchronise it?'
for i, title, _, rel in retrieve(mfa_q):
    print(f'  {rel:.4f}  {i}  {title[:52]}')

  0.6901  KB-025  Two-factor token out of sync
  0.6014  KB-006  Multi-factor authentication push notification never 
  0.4080  KB-017  Mobile device shows 'device not compliant'
  0.3950  KB-008  Password reset self-service portal rejects the new p


#run all three contradictions, three times each


In [60]:
CONTRADICTIONS = {
 'MFA token  ': mfa_q,
 'OneDrive   ': 'OneDrive is stuck syncing and the user needs a file back. Should I delete the local sync folder?',
 'loan device': 'A laptop has a mainboard fault. Can I give the user a loan device from the local cupboard?',
}

for label, q in CONTRADICTIONS.items():
    for run in range(3):
        s = answer(q)
        print(f'{label} run {run}: gate={s["gate"]:9} '
              f'abstained={str(s["abstained"]):5} cited={s["citations"]}')
    print()

MFA token   run 0: gate=model     abstained=True  cited=[]
MFA token   run 1: gate=model     abstained=True  cited=[]
MFA token   run 2: gate=model     abstained=True  cited=[]

OneDrive    run 0: gate=answered  abstained=False cited=['RB-06']
OneDrive    run 1: gate=answered  abstained=False cited=['KB-012', 'RB-06']
OneDrive    run 2: gate=answered  abstained=False cited=['RB-06']

loan device run 0: gate=model     abstained=True  cited=[]
loan device run 1: gate=model     abstained=True  cited=[]
loan device run 2: gate=model     abstained=True  cited=[]

